maturin + mm0-rs
just do python version from scratch
import vs export

my datalog provenance post https://www.philipzucker.com/metamath-datalog-provenenance/

https://digama0.github.io/mm0/thesis.pdf thesis

https://github.com/garrettkatz/mmpy
https://github.com/david-a-wheeler/mmverify.py

metamath hare?
https://github.com/gleachkr/mm0-zig

sudo apt-get install metamath

https://github.com/digama0/mm0/blob/master/mm0.md

https://grahamlk.me/Aufbau/#hilbert

Hmm. The variable condition is goofy enough, I wonder if a regular egraph could handle it.

rpython? it kind of is a bytecode interpreter. Hmm. Not really loops though, so like would the jit ever kick in? Prob not. You like run through in a single pass.


@relation
@congruence



In [2]:
%%file /tmp/hilbert.mm0
delimiter $ ( ) $;
provable sort wff;
term imp (a b: wff): wff; infixr imp: $->$ prec 25;
term not (a: wff): wff; prefix not: $not$ prec 35;
axiom h1 (a b: wff): $ a -> (b -> a) $;
axiom h2 (a b c: wff):
  $ (a -> (b -> c)) -> ((a -> b) -> (a -> c)) $;
axiom h3 (a b: wff): $ (not a -> not b) -> (b -> a) $;
axiom con2 (a b: wff): $ (a -> not b) -> (b -> not a) $;
axiom inot (a: wff): $ (a -> not a) -> not a $;
axiom mp (a b: wff): $ a $ > $ a -> b $ > $ b $;
theorem imp_refl (a: wff): $ a -> a $;
theorem hs (a b c: wff): $ a -> b $ > $ b -> c $ > $ a -> c $;
theorem notnot1 (p: wff): $ p -> not not p $;
theorem dne (p: wff): $ not not p -> p $;


Overwriting /tmp/hilbert.mm0


In [ ]:
from dataclasses import dataclass,field
@dataclass
class Sort:
    name : str
    provable : bool = False

wff = Sort("wff", provable=True)

@dataclass
class Decl:
    name : str
    args : list[Sort]
    out : Sort
    #bound_args : list[Sort] = field(default_factory=list)
    #args : list[Sort] = field(default_factory=list)# name,sort tuples?

    def __call__(self, *args):
        # type check
        if len(args) != len(self.args):
            raise TypeError(f"{self.name} takes {len(self.args)} arguments, got {len(args)}")
        for i, (arg, sort) in enumerate(zip(args, self.args)):
            if arg.sort != sort:
                raise TypeError(f"Argument {i} of {self.name} should be of sort {sort.name}, got {arg.sort.name}")
        return App(self, list(args)) 
    

imp = Decl("imp", [wff, wff], [wff])


# App is a judgement
@dataclass
class App:
    decl : Decl
    args : list["Term"]

    @property
    def sort(self):
        return self.decl.out

    @property(self)
    def vars(self):
        return set.union(*[arg.vars for arg in self.args])


@dataclass
class Var:
    name : str
    sort : Sort

    @property
    def vars(self):
        return set([self])

@dataclass
class BVar:
    name : str
    sort : Sort

    @property
    def vars(self):
        return set([self])


type Term = App | Var | BVar

def subst(t : Term, substs : dict[Var, Term]) -> Term:
    if isinstance(t, Var):
        return substs.get(t, t)
    elif isinstance(t, App):
        return App(t.decl, [subst(arg, substs) for arg in t.args])
    elif isinstance(t, BVar):
        return t
    else:
        raise TypeError(f"Unknown term type: {type(t)}")



@dataclass
class Assertion:
    name : str
    bvars : list[Decl]
    vars : list[Decl]
    hyps : list[Term]
    body : Term
    proof : ProofTerm | None
    #proof : AppAssertion #list[tuple[Term, tuple[]]]

    # check?
    def __call__(self, *args):
        return PfApp(self, args)
    
    def check(self):
        if self.proof is None:
            return True
        else:
            

        pass
    def __post_init__(self):
        assert check


@dataclass
class PfApp:
    assertion : Assertion
    args : list[Term]
    assertargs : list[ProofTerm] 

    def check(self, ctx):
        # check is valid proof of assertion

    def conc(self):
        subst(self.assertion.body, {var.name: arg for var, arg in zip(self.assertion.vars, self.args)})


type ProofTerm = PfApp | int


In [ ]:



"""
@dataclass
class AppAssertion:
    ctx: list[Term]
    assertion : Proof | int
    args : list[Term]
    assertargs : list[AppAssertions] #?
"""


@dataclass
class Axiom:
    pass
    #assertion : Assertion

@dataclass
class ProofTerm:
    assertion : Assertion

type Proof = ProofTerm | Axiom

#@dataclass
#class Proof

def axiom(name, vars, bvars=[]):
    return Assertion(name, vars, bvars, proof=None)

def theorem(name, vars, proof, bvars=[]):
    # check proof?
    return Assertion(name, vars, bvars, proof=proof)

def subst(ctx : list[Var], a : Assertion, *subst):
    # 
    

If defns are just unfolded, maybe I could just use python meta functions as defns

I could implement this over the z3 ast with some meta carrying around what is bvar
But I dunno if the bvar notion would play right?


